[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/C.%20Transportation%2C%20Routing%20%26%20Freight%20Management%20%28The%20Physical%20Internet%29/077.%20The%20Dynamic%20%26%20Real-Time%20VRP/P77-Tier-9_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---

# 77. The Dynamic and Real-Time Vehicle Routing Problem
## Tier 9: QUBO Formulation and Annealing-Based Search

This notebook builds a quadratic unconstrained binary optimization (QUBO) model for assigning requests to vehicles, then searches for a low-energy binary assignment using a simulated annealing procedure. The scenario contains vehicles, requests, vehicle capacities, request time windows, and a current planning time. The outputs are the QUBO coefficients, a candidate assignment, an assignment-coverage score, and a sequence of small request-arrival experiments.

### Scope and modeling assumptions

The method represents each vehicle-request assignment with one binary variable. It models the requirement that each request be assigned to exactly one vehicle, and adds penalties for assignments that violate an individual vehicle-capacity check or the request's modeled time window. Travel cost is a Euclidean-distance proxy, normalized by 100 distance units so its scale is compatible with the assignment penalty.

This is an assignment model, not a complete route-sequencing model. It does not enforce cumulative vehicle capacity across multiple requests, vehicle-route continuity, service times, pickup-and-delivery precedence, or interactions between the routes of different vehicles. A request's arrival time is approximated by the distance from the vehicle's current position to the request origin, with one distance unit treated as one time unit. The generated time windows are relative to the planning horizon; the displayed clock time is scenario context and is not added to those relative windows.

The simulator below is a classical simulated annealing heuristic operating on binary assignments. It does not represent quantum amplitudes, simulate a quantum circuit, or establish quantum speedup, optimality, uncertainty quantification, or hardware performance. The names of the data structures retain the chapter's quantum-themed terminology, but their implementation and interpretation are stated explicitly here.

### Solution sequence

1. Construct a dynamic vehicle-routing scenario.
2. Build a QUBO objective and its assignment and feasibility penalties.
3. Search the binary state space with a temperature-scheduled Metropolis procedure.
4. Decode and inspect the resulting vehicle-request assignments.
5. Repeat the same modeling and search steps as new requests arrive.

### QUBO model

Let \(i \in \{0,\ldots,V-1\}\) index vehicles and \(j \in \{0,\ldots,R-1\}\) index requests. The binary variable \(x_{ij}\) is 1 when vehicle \(i\) is assigned request \(j\), and 0 otherwise. The travel proxy \(c_{ij}\) is the sum of the vehicle-to-origin, origin-to-destination, and destination-to-depot Euclidean distances, divided by 100. The assignment penalty is \(\lambda_A\); infeasible capacity and time-window assignments receive penalties \(\lambda_C\) and \(\lambda_T\), respectively.

$$
E(x) = \sum_{i=0}^{V-1}\sum_{j=0}^{R-1} c_{ij}x_{ij}
+ \lambda_A\sum_{j=0}^{R-1}\left(1-\sum_{i=0}^{V-1}x_{ij}\right)^2
+ \lambda_C\sum_{(i,j)\in F_C}x_{ij}
+ \lambda_T\sum_{(i,j)\in F_T}x_{ij}.
$$

Here, \(F_C\) is the set of vehicle-request pairs that fail the individual capacity check, and \(F_T\) is the set of pairs whose estimated arrival falls outside the request's time window. The squared assignment term is zero when a request has exactly one assigned vehicle, positive when it is unassigned or assigned to multiple vehicles, and expands into linear and pairwise terms. The QUBO omits the constant \(\lambda_A R\), which does not affect which binary assignment minimizes energy. The code stores the linear coefficients in `linear_coefficients` and the pairwise coefficients in `quadratic_coefficients`.

The search begins with all assignment variables set to zero. At each proposal it flips one variable, computes the resulting energy change, and accepts an improving move or accepts a worsening move with probability \(\exp(-\beta\Delta E)\), where \(\beta=1/T\). This temperature schedule is a heuristic search mechanism; it does not guarantee a global optimum. A returned assignment-coverage score measures the fraction of requests assigned exactly once. It is not a percentage of optimality.

In [ ]:
from typing import Dict, List, Optional, Tuple
from dataclasses import dataclass
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# Fixed seeds make the generated scenario and annealing proposals reproducible.
np.random.seed(42)
random.seed(42)

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
@dataclass
class QuantumDVRPState:
    """Input data for one dynamic vehicle-routing planning state."""

    vehicles: List[Dict]
    requests: List[Dict]
    time: float
    capacity_constraints: List[int]
    time_windows: List[Tuple[float, float]]


@dataclass
class QUBOFormulation:
    """Linear and pairwise coefficients for a binary QUBO objective."""

    num_vehicles: int
    num_requests: int
    num_variables: int
    linear_coefficients: np.ndarray
    quadratic_coefficients: Dict[Tuple[int, int], float]
    penalty_weights: Dict[str, float]

    def __post_init__(self):
        expected_variables = self.num_vehicles * self.num_requests
        if self.num_variables != expected_variables:
            raise ValueError("The variable count must equal vehicles times requests.")
        if len(self.linear_coefficients) != expected_variables:
            raise ValueError("The linear coefficient vector has an inconsistent length.")


@dataclass
class QuantumState:
    """A classical binary candidate used by the annealing simulator."""

    binary_state: np.ndarray
    num_variables: int
    energy: float = 0.0

    def __post_init__(self):
        if self.binary_state is None:
            self.binary_state = np.zeros(self.num_variables, dtype=np.int8)
        else:
            self.binary_state = np.asarray(self.binary_state, dtype=np.int8)
        if self.binary_state.shape != (self.num_variables,):
            raise ValueError("The binary candidate has an inconsistent shape.")


@dataclass
class QuantumAnnealingResult:
    """Search result; the probability distribution is not estimated."""

    best_solution: np.ndarray
    best_energy: float
    solution_quality: float
    computation_time: float
    convergence_iteration: int
    probability_distribution: Optional[np.ndarray] = None

## Build the QUBO coefficients

The QUBO builder translates the model into the coefficient convention used by the energy evaluator:

$$
E(x)=\sum_k a_kx_k+\sum_{k<\ell}b_{k\ell}x_kx_\ell.
$$

For each request, expanding \(\lambda_A(1-\sum_i x_{ij})^2\) gives a linear contribution of \(-\lambda_A\) for each vehicle assignment variable and a pairwise contribution of \(2\lambda_A\) for each pair of distinct vehicles. The omitted constant \(\lambda_A\) per request does not change the minimizing assignment. Capacity and time-window penalties are added to the linear coefficient of an infeasible assignment variable.

In [ ]:
class QUBOBuilder:
    """Build a binary assignment QUBO from a dynamic routing state."""

    def __init__(self, penalty_weights: Optional[Dict[str, float]] = None):
        self.penalty_weights = penalty_weights or {
            "capacity": 1000.0,
            "time_window": 500.0,
            "assignment": 10.0,
            "routing": 1.0,
        }

    def build_qubo(self, dvrp_state: QuantumDVRPState) -> QUBOFormulation:
        """Encode travel cost, exact assignment, and pairwise feasibility."""
        num_vehicles = len(dvrp_state.vehicles)
        num_requests = len(dvrp_state.requests)
        if len(dvrp_state.capacity_constraints) != num_vehicles:
            raise ValueError("There must be one capacity constraint per vehicle.")
        if len(dvrp_state.time_windows) != num_requests:
            raise ValueError("There must be one time window per request.")

        num_variables = num_vehicles * num_requests
        linear_coeffs = np.zeros(num_variables, dtype=float)
        quadratic_coeffs: Dict[Tuple[int, int], float] = {}

        assignment_penalty = self.penalty_weights["assignment"]
        for i, vehicle in enumerate(dvrp_state.vehicles):
            for j, request in enumerate(dvrp_state.requests):
                var_idx = i * num_requests + j

                distance_to_origin = self._calculate_distance(
                    vehicle["position"], request["origin"]
                )
                distance_to_destination = self._calculate_distance(
                    request["origin"], request["destination"]
                )
                distance_to_depot = self._calculate_distance(
                    request["destination"], (0, 0)
                )

                # Normalize the distance proxy so assignment penalties have a comparable scale.
                travel_cost = (
                    distance_to_origin + distance_to_destination + distance_to_depot
                ) / 100.0
                linear_coeffs[var_idx] += travel_cost - assignment_penalty

                arrival_estimate = distance_to_origin
                window_start, window_end = dvrp_state.time_windows[j]
                if request["demand"] > dvrp_state.capacity_constraints[i]:
                    linear_coeffs[var_idx] += self.penalty_weights["capacity"]
                if not window_start <= arrival_estimate <= window_end:
                    linear_coeffs[var_idx] += self.penalty_weights["time_window"]

        # These pairwise terms penalize assigning one request to multiple vehicles.
        for j in range(num_requests):
            for i in range(num_vehicles):
                first_idx = i * num_requests + j
                for k in range(i + 1, num_vehicles):
                    second_idx = k * num_requests + j
                    quadratic_coeffs[(first_idx, second_idx)] = 2.0 * assignment_penalty

        return QUBOFormulation(
            num_vehicles=num_vehicles,
            num_requests=num_requests,
            num_variables=num_variables,
            linear_coefficients=linear_coeffs,
            quadratic_coefficients=quadratic_coeffs,
            penalty_weights=self.penalty_weights.copy(),
        )

    def _calculate_distance(
        self, pos1: Tuple[float, float], pos2: Tuple[float, float]
    ) -> float:
        """Return Euclidean distance for the model's travel-cost proxy."""
        return float(np.hypot(pos1[0] - pos2[0], pos1[1] - pos2[1]))

## Search the QUBO with simulated annealing

A binary state is a complete candidate assignment: each bit records whether one vehicle serves one request. A proposal flips one bit, so it changes only that assignment decision. The simulator evaluates the energy change using the flipped variable's linear coefficient and its pairwise interactions; this avoids allocating an array with \(2^n\) amplitudes for \(n\) binary variables.

At each temperature, the Metropolis rule accepts a move with energy change \(\Delta E\) if it improves the objective, or with probability \(\exp(-\Delta E/T)\) if it worsens it. Cooling gradually reduces acceptance of worsening moves. The returned best state is the lowest-energy state encountered during this heuristic search, not a proof of global optimality.

The QUBO has \(n=VR\) binary variables. Its assignment interactions contain up to \(R\binom{V}{2}\) pairwise terms. With a sparse interaction lookup, each single-variable proposal examines only interactions involving that variable. The search memory is proportional to the number of variables plus stored interactions; it does not store all \(2^n\) candidate states.

In [ ]:
class QuantumSimulator:
    """Run a classical Metropolis simulated annealing search on a QUBO."""

    def __init__(
        self,
        num_qubits: int,
        temperature_schedule: Optional[List[float]] = None,
    ):
        # The parameter name is retained for continuity; it counts binary variables, not physical qubits.
        self.num_variables = num_qubits
        self.temperature_schedule = (
            temperature_schedule or self._default_temperature_schedule()
        )
        if any(temperature <= 0 for temperature in self.temperature_schedule):
            raise ValueError("Every annealing temperature must be positive.")
        self.beta_schedule = [1.0 / temperature for temperature in self.temperature_schedule]

    def _default_temperature_schedule(self) -> List[float]:
        """Return the exponential cooling schedule used by the search."""
        return [10.0 * (0.95 ** i) for i in range(100)]

    def initialize_state(self, qubo: QUBOFormulation) -> QuantumState:
        """Start from the all-zero assignment and let the search add assignments."""
        return QuantumState(
            num_variables=qubo.num_variables,
            binary_state=np.zeros(qubo.num_variables, dtype=np.int8),
        )

    def calculate_energy(self, state: QuantumState, qubo: QUBOFormulation) -> float:
        """Evaluate the linear and sparse pairwise QUBO terms."""
        linear_energy = float(np.dot(qubo.linear_coefficients, state.binary_state))
        quadratic_energy = sum(
            coefficient * state.binary_state[i] * state.binary_state[j]
            for (i, j), coefficient in qubo.quadratic_coefficients.items()
        )
        return linear_energy + quadratic_energy

    def quantum_annealing(self, qubo: QUBOFormulation) -> QuantumAnnealingResult:
        """Search binary assignments with single-variable Metropolis proposals."""
        if qubo.num_variables != self.num_variables:
            raise ValueError("The simulator and QUBO must have the same variable count.")
        if qubo.num_variables == 0:
            raise ValueError("The QUBO must contain at least one binary variable.")

        start_time = time.time()
        current_state = self.initialize_state(qubo)
        current_energy = self.calculate_energy(current_state, qubo)
        best_state = current_state.binary_state.copy()
        best_energy = current_energy
        iteration = 0

        interactions: List[List[Tuple[int, float]]] = [
            [] for _ in range(qubo.num_variables)
        ]
        for (i, j), coefficient in qubo.quadratic_coefficients.items():
            interactions[i].append((j, coefficient))
            interactions[j].append((i, coefficient))

        for iteration, (temperature, beta) in enumerate(
            zip(self.temperature_schedule, self.beta_schedule), start=1
        ):
            for _ in range(10):
                variable = random.randrange(qubo.num_variables)
                old_value = int(current_state.binary_state[variable])
                delta_energy = qubo.linear_coefficients[variable] * (1 - 2 * old_value)
                delta_energy += sum(
                    coefficient
                    * current_state.binary_state[other_variable]
                    * (1 - 2 * old_value)
                    for other_variable, coefficient in interactions[variable]
                )

                if delta_energy <= 0 or random.random() < np.exp(-beta * delta_energy):
                    current_state.binary_state[variable] = 1 - old_value
                    current_energy += float(delta_energy)

                    if current_energy < best_energy:
                        best_energy = current_energy
                        best_state = current_state.binary_state.copy()

        elapsed = time.time() - start_time
        coverage = self._estimate_solution_quality(best_state, qubo)
        return QuantumAnnealingResult(
            best_solution=best_state,
            best_energy=float(best_energy),
            solution_quality=coverage,
            computation_time=elapsed,
            convergence_iteration=iteration,
            probability_distribution=None,
        )

    def _estimate_solution_quality(
        self, solution: np.ndarray, qubo: QUBOFormulation
    ) -> float:
        """Measure the fraction of requests assigned to exactly one vehicle."""
        if qubo.num_requests == 0:
            return 1.0
        exactly_once = 0
        for j in range(qubo.num_requests):
            assigned_vehicles = sum(
                solution[i * qubo.num_requests + j]
                for i in range(qubo.num_vehicles)
            )
            if assigned_vehicles == 1:
                exactly_once += 1
        return exactly_once / qubo.num_requests

## Create the dynamic scenario

The scenario keeps the vehicle and request counts, capacity ranges, coordinate ranges, demand ranges, deadlines, priorities, and random seeds specified for this example. The reported \(2^{VR}\) is the number of possible binary assignment vectors for \(VR\) variables; it describes the mathematical search space and is not enumerated by the simulator.

In [ ]:
def create_quantum_scenario() -> Dict:
    """Create the vehicle, request, and planning-horizon data."""
    print("CREATING THE DYNAMIC ROUTING SCENARIO")
    print("=" * 42)

    vehicles = []
    for i in range(47):
        vehicles.append(
            {
                "id": f"V{i + 1}",
                "position": (random.randint(0, 100), random.randint(0, 100)),
                "capacity": random.randint(12, 45),
                "utilization": random.uniform(0.3, 0.9),
            }
        )

    requests = []
    for i in range(180):
        requests.append(
            {
                "id": f"R{i + 1}",
                "origin": (random.randint(0, 100), random.randint(0, 100)),
                "destination": (random.randint(0, 100), random.randint(0, 100)),
                "demand": random.randint(1, 5),
                "deadline": random.uniform(15, 120),
                "priority": random.choices([1, 2, 3], weights=[0.2, 0.5, 0.3])[0],
            }
        )

    capacity_constraints = [vehicle["capacity"] for vehicle in vehicles]
    time_windows = [
        (request["deadline"] - 30, request["deadline"] + 30)
        for request in requests
    ]
    current_time = 14 * 60 + 47

    print(f"Vehicles: {len(vehicles)}")
    print(f"Requests: {len(requests)}")
    print(f"Binary assignment variables: {len(vehicles) * len(requests)}")
    print(f"Binary search-space size: 2^{len(vehicles) * len(requests)}")
    print(f"Displayed planning clock time: {current_time // 60:02d}:{current_time % 60:02d}")

    return {
        "vehicles": vehicles,
        "requests": requests,
        "capacity_constraints": capacity_constraints,
        "time_windows": time_windows,
        "time": current_time,
        "num_vehicles": len(vehicles),
        "num_requests": len(requests),
    }


quantum_scenario = create_quantum_scenario()

## Build, optimize, and decode

The optimization function follows the model's sequence: package the scenario as a routing state, build its QUBO, run the annealing search, and decode every selected binary variable into a vehicle-request pair. Decoded costs use the same normalized Euclidean travel proxy as the QUBO objective. The reported coverage is the fraction of requests assigned exactly once; the result does not claim a known optimum or compare against unsupported benchmark timings.

In [ ]:
def quantum_optimize_dvrp(
    quantum_scenario: Dict, classical_comparison: bool = True
) -> Dict:
    """Build and search the scenario's QUBO, then decode its assignments."""
    del classical_comparison  # No benchmark result is available for a valid comparison.

    dvrp_state = QuantumDVRPState(
        vehicles=quantum_scenario["vehicles"],
        requests=quantum_scenario["requests"],
        time=quantum_scenario["time"],
        capacity_constraints=quantum_scenario["capacity_constraints"],
        time_windows=quantum_scenario["time_windows"],
    )

    qubo = QUBOBuilder().build_qubo(dvrp_state)
    print("QUBO formulation built")
    print(f"Binary variables: {qubo.num_variables}")
    print(f"Linear coefficients: {len(qubo.linear_coefficients)}")
    print(f"Pairwise coefficients: {len(qubo.quadratic_coefficients)}")

    quantum_sim = QuantumSimulator(num_qubits=qubo.num_variables)
    result = quantum_sim.quantum_annealing(qubo)
    assignments = decode_quantum_solution(result.best_solution, quantum_scenario)

    print(f"Best QUBO energy: {result.best_energy:.3f}")
    print(f"Assignment coverage: {result.solution_quality:.1%}")
    print(f"Annealing search time: {result.computation_time:.3f} seconds")
    print(f"Cooling-schedule steps: {result.convergence_iteration}")

    return {
        "quantum_result": result,
        "assignments": assignments,
        "optimization_time": result.computation_time,
        "num_vehicles": quantum_scenario["num_vehicles"],
        "num_requests": quantum_scenario["num_requests"],
        "solution_space_size": 2 ** qubo.num_variables,
    }


def decode_quantum_solution(binary_solution: np.ndarray, scenario: Dict) -> List[Dict]:
    """Convert selected binary variables into vehicle-request records."""
    assignments = []
    num_vehicles = scenario["num_vehicles"]
    num_requests = scenario["num_requests"]

    if len(binary_solution) != num_vehicles * num_requests:
        raise ValueError("The binary solution length does not match the scenario.")

    for i in range(num_vehicles):
        vehicle = scenario["vehicles"][i]
        for j in range(num_requests):
            var_idx = i * num_requests + j
            if binary_solution[var_idx] == 1:
                request = scenario["requests"][j]
                distance = float(
                    np.hypot(
                        vehicle["position"][0] - request["origin"][0],
                        vehicle["position"][1] - request["origin"][1],
                    )
                    + np.hypot(
                        request["origin"][0] - request["destination"][0],
                        request["origin"][1] - request["destination"][1],
                    )
                    + np.hypot(request["destination"][0], request["destination"][1])
                )
                assignments.append(
                    {
                        "vehicle_id": vehicle["id"],
                        "request_id": request["id"],
                        "cost": distance / 100.0,
                        "deadline": scenario["time_windows"][j][1],
                        "priority": request["priority"],
                    }
                )
    return assignments

## Model request arrivals over time

Each experiment step starts from a copy of the scenario state, adds the requests defined for that step, updates its time-window list, and solves the resulting QUBO. Copying the state keeps the original scenario intact and makes each step's input explicit. These repeated solves illustrate re-optimization after arrivals; they do not implement continuous-time adaptation or reuse a quantum state.

In [ ]:
def simulate_dynamic_adaptation() -> List[Dict]:
    """Re-optimize copied scenario states as requests arrive."""
    adaptation_results = []
    evolving_scenario = quantum_scenario

    for step in range(5):
        evolving_scenario = modify_scenario_for_step(evolving_scenario, step)
        print(f"\nPlanning step {step + 1}/5")
        step_results = quantum_optimize_dvrp(
            evolving_scenario, classical_comparison=False
        )
        record = {
            "step": step + 1,
            "num_requests": evolving_scenario["num_requests"],
            "optimization_time": step_results["optimization_time"],
            "solution_quality": step_results["quantum_result"].solution_quality,
            "assigned_requests": len(step_results["assignments"]),
        }
        adaptation_results.append(record)
        print(f"Requests in planning state: {record['num_requests']}")
        print(f"Requests assigned: {record['assigned_requests']}")
        print(f"Assignment coverage: {record['solution_quality']:.1%}")

    return adaptation_results


def modify_scenario_for_step(scenario: Dict, step: int) -> Dict:
    """Return a copied scenario with the arrivals for one planning step."""
    modified_scenario = {
        key: list(value) if isinstance(value, list) else value
        for key, value in scenario.items()
    }
    modified_scenario["vehicles"] = [dict(vehicle) for vehicle in scenario["vehicles"]]
    modified_scenario["requests"] = [dict(request) for request in scenario["requests"]]

    arrivals_by_step = {1: 8, 2: 5, 3: 3, 4: 2}
    arrival_count = arrivals_by_step.get(step, 0)
    for _ in range(arrival_count):
        request_number = len(modified_scenario["requests"]) + 1
        if step == 3:
            deadline = random.uniform(15, 45)
            priority = 1
        elif step == 4:
            deadline = random.uniform(20, 60)
            priority = 1
        elif step == 1:
            deadline = random.uniform(15, 120)
            priority = random.choices([1, 2, 3], weights=[0.3, 0.4, 0.3])[0]
        elif step == 2:
            deadline = random.uniform(15, 120)
            priority = random.choices([1, 2, 3], weights=[0.2, 0.6, 0.2])[0]
        else:
            deadline = random.uniform(15, 120)
            priority = random.choices([1, 2, 3], weights=[0.2, 0.5, 0.3])[0]

        modified_scenario["requests"].append(
            {
                "id": f"R{request_number}",
                "origin": (random.randint(0, 100), random.randint(0, 100)),
                "destination": (random.randint(0, 100), random.randint(0, 100)),
                "demand": random.randint(1, 5),
                "deadline": deadline,
                "priority": priority,
            }
        )

    modified_scenario["num_requests"] = len(modified_scenario["requests"])
    modified_scenario["time_windows"] = [
        (request["deadline"] - 30, request["deadline"] + 30)
        for request in modified_scenario["requests"]
    ]
    modified_scenario["time"] += 15.0
    return modified_scenario

## Visualize assignments and repeated planning

The plots describe the computed candidate assignments and the repeated planning runs. They do not compare the heuristic with a classical benchmark or interpret the binary search-space size as a measured runtime. The assignment-coverage panel uses a percentage, the request-arrival panel uses request counts, and the vehicle panel counts decoded assignments.

In [ ]:
def visualize_quantum_results(
    quantum_results: Dict, adaptation_results: List[Dict]
) -> None:
    """Plot candidate assignments and the request-arrival experiments."""
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle("QUBO Assignment Search and Dynamic Planning", fontsize=15)

    assignments = quantum_results["assignments"]
    vehicle_ids = [vehicle["id"] for vehicle in quantum_scenario["vehicles"]]
    counts_by_vehicle = {vehicle_id: 0 for vehicle_id in vehicle_ids}
    for assignment in assignments:
        counts_by_vehicle[assignment["vehicle_id"]] += 1

    axes[0, 0].bar(
        list(counts_by_vehicle.keys()), list(counts_by_vehicle.values())
    )
    axes[0, 0].set_title("Decoded Assignments by Vehicle")
    axes[0, 0].set_xlabel("Vehicle")
    axes[0, 0].set_ylabel("Assigned requests (count)")
    axes[0, 0].tick_params(axis="x", rotation=90)
    axes[0, 0].grid(True, alpha=0.3, axis="y")

    covered_requests = len({assignment["request_id"] for assignment in assignments})
    request_counts = [
        covered_requests,
        quantum_results["num_requests"] - covered_requests,
    ]
    axes[0, 1].bar(["Assigned at least once", "Not assigned"], request_counts)
    axes[0, 1].set_title("Request Assignment Coverage")
    axes[0, 1].set_ylabel("Requests (count)")
    axes[0, 1].tick_params(axis="x", rotation=15)
    axes[0, 1].grid(True, alpha=0.3, axis="y")

    steps = [record["step"] for record in adaptation_results]
    coverage = [100 * record["solution_quality"] for record in adaptation_results]
    axes[1, 0].plot(steps, coverage, marker="o", linewidth=2, label="Exactly-once coverage")
    axes[1, 0].set_title("Assignment Coverage Across Planning Steps")
    axes[1, 0].set_xlabel("Planning step")
    axes[1, 0].set_ylabel("Requests assigned exactly once (%)")
    axes[1, 0].set_ylim(0, 100)
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    request_counts_by_step = [record["num_requests"] for record in adaptation_results]
    axes[1, 1].plot(
        steps,
        request_counts_by_step,
        marker="s",
        linewidth=2,
        color="tab:orange",
        label="Requests in planning state",
    )
    axes[1, 1].set_title("Requests in Each Planning State")
    axes[1, 1].set_xlabel("Planning step")
    axes[1, 1].set_ylabel("Requests (count)")
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)

    fig.tight_layout()
    plt.show()

## Run the scenario and inspect the result

The main run prints the candidate's energy, assignment coverage, and decoded assignments, then carries out the five request-arrival planning steps and displays the plots. Because the search is heuristic, its outcome depends on the search path and should be interpreted as a candidate solution under the stated model assumptions.

In [ ]:
print("STARTING QUBO-BASED DYNAMIC ROUTING SEARCH")
print("=" * 46)

quantum_results = quantum_optimize_dvrp(
    quantum_scenario, classical_comparison=False
)

print("\nCandidate assignment summary")
print(f"Vehicles: {quantum_results['num_vehicles']}")
print(f"Requests: {quantum_results['num_requests']}")
print(f"Decoded vehicle-request assignments: {len(quantum_results['assignments'])}")
print(f"Exactly-once assignment coverage: {quantum_results['quantum_result'].solution_quality:.1%}")
print(f"Best QUBO energy: {quantum_results['quantum_result'].best_energy:.3f}")

print("\nSample decoded assignments")
for index, assignment in enumerate(quantum_results["assignments"][:10], start=1):
    print(
        f"{index:2d}. {assignment['vehicle_id']} serves "
        f"{assignment['request_id']} | normalized distance cost: "
        f"{assignment['cost']:.3f} | priority: {assignment['priority']} | "
        f"window end: {assignment['deadline']:.1f}"
    )

adaptation_results = simulate_dynamic_adaptation()
visualize_quantum_results(quantum_results, adaptation_results)

## Interpretation and limitations

The best energy is the value of the encoded QUBO objective for the best candidate encountered by the search. Assignment coverage reports how many requests have exactly one selected vehicle; a value below 100% indicates that at least one request is either unassigned or assigned more than once. The decoded assignment list may contain multiple assignments for a request, so its length is not itself a count of uniquely served requests.

The scenario's binary search space has \(2^{VR}\) candidates, but this fact alone does not establish the runtime or scalability of an optimization method. The implemented search checks a limited sequence of single-bit proposals and provides no optimality certificate. The penalty formulation also relies on its chosen coefficients, so a low-energy candidate is not automatically proof that every operational constraint is satisfied.

The implementation is a classical heuristic demonstration of QUBO construction and annealing-based search. It does not provide quantum hardware results, quantum speedup measurements, a probability distribution over solutions, or a benchmark comparison. More detailed routing realism would require extending the model with route order, cumulative capacity, travel and service times, and any operational constraints relevant to a particular application.